# 10年定着予測 - Dブロック拡張 + 複数モデル比較（CatBoost/LightGBM/XGBoost）

**背景**: `17_tfidf_tuning_and_combos`で、TF-IDF(A_v1, 15次元) + 四半期/加速度特徴量(D, 6指標)の組み合わせが
Public 0.552540で総合最良を更新した。今回は以下3点を実施する。

## 実施すること

1. **Dブロックの拡張**: 現状6指標（残業時間・欠勤日数・月例給与・360度評価_親和度・有給取得日数・研修時間）を、
   月次データの全16指標に拡張し、改善するか確認する。
2. **複数splitでの検証を標準プロセスとして採用**: `17_`で単一分割がTF-IDFの効果を過大評価していたことが
   判明したため、以降は常に80/20と75/25の2つの時系列splitで評価する。
3. **複数モデルでの比較**: CatBoost（継続検証）に加え、LightGBM・XGBoostでも同じ特徴量レシピを検証する。
   将来的なアンサンブルを見据え、各モデル単体の性能を比較した上で、**単純平均アンサンブル**
   （学習された重みを使う方式は過学習しやすいという既知の教訓を踏まえ、あえて固定ヒューリスティックのみ採用）も試す。

## 長時間実行への対策（チェックポイント機能）

前回`17_`の実行中にColabとの接続が切れる事象があったため、各設定の結果を計算次第すぐにCSVへ追記し、
**再実行時に既に計算済みの設定は自動的にスキップ**する仕組みを導入する。接続が切れても、
同じセルから再実行すれば完了済みの分は再計算されない。

## 次のアクション（今回は着手しない、将来のバックログ）

- テキスト特徴量を文字n-gram TF-IDFから日本語の事前学習済み文埋め込みモデルに置き換える案

## 実行環境
Google Colab（GPU: T4）を想定。

In [1]:
!pip install -q catboost lightgbm xgboost optuna

In [2]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

Sun Aug  9 12:13:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P0             27W /   70W |     109MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import sys
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path("/content/drive/MyDrive/jaggle_2026")
sys.path.append(str(PROJECT_ROOT))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import datetime
import warnings

import numpy as np
import pandas as pd
import catboost as cb
import lightgbm as lgb
import xgboost as xgb
import optuna
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import log_loss
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

from common.utils.logger import get_logger
from common.utils.metrics import calculate_logloss
from common.utils.seed import seed_everything

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
seed_everything(seed=SEED)

TARGET_COL = "10年定着ラベル"
ID_COL = "社員ID"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [5]:
SCRIPT_NAME = "18_multi_model_expanded_d"
TODAY = datetime.datetime.now().strftime("%Y%m%d")

LOG_DIR = PROJECT_ROOT / "logs"
logger = get_logger(SCRIPT_NAME, log_dir=str(LOG_DIR))
logger.info(f"=== [{SCRIPT_NAME}] 実験開始 ===")

OUTPUT_DIR = PROJECT_ROOT / "data" / "output" / TODAY
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAVED_MODELS_DIR = PROJECT_ROOT / "saved_models" / TODAY / SCRIPT_NAME
SAVED_MODELS_DIR.mkdir(parents=True, exist_ok=True)

# チェックポイント（日付非依存の固定パス。セッションをまたいで再開できるようにする）
CHECKPOINT_DIR = PROJECT_ROOT / "data" / "output" / "_checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = CHECKPOINT_DIR / f"{SCRIPT_NAME}_checkpoint.csv"

logger.info(f"Output Directory: {OUTPUT_DIR}")
logger.info(f"Checkpoint Path: {CHECKPOINT_PATH}")
if CHECKPOINT_PATH.exists():
    logger.info(f"既存のチェックポイントを発見: {len(pd.read_csv(CHECKPOINT_PATH))}件の結果が記録済み")
else:
    logger.info("チェックポイントは未作成（新規実行）")

[2026-08-09 12:13:11] [INFO] === [18_multi_model_expanded_d] 実験開始 ===


INFO:18_multi_model_expanded_d:=== [18_multi_model_expanded_d] 実験開始 ===


[2026-08-09 12:13:11] [INFO] Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260809


INFO:18_multi_model_expanded_d:Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260809


[2026-08-09 12:13:11] [INFO] Checkpoint Path: /content/drive/MyDrive/jaggle_2026/data/output/_checkpoints/18_multi_model_expanded_d_checkpoint.csv


INFO:18_multi_model_expanded_d:Checkpoint Path: /content/drive/MyDrive/jaggle_2026/data/output/_checkpoints/18_multi_model_expanded_d_checkpoint.csv


[2026-08-09 12:13:11] [INFO] チェックポイントは未作成（新規実行）


INFO:18_multi_model_expanded_d:チェックポイントは未作成（新規実行）


In [6]:
INPUT_DIR = PROJECT_ROOT / "data" / "input"

train_persona = pd.read_csv(INPUT_DIR / "employee_persona_train.csv")
test_persona = pd.read_csv(INPUT_DIR / "employee_persona_test.csv")
train_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_train.csv")
test_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_test.csv")

logger.info(f"Train Persona Shape: {train_persona.shape}, Test Persona Shape: {test_persona.shape}")
logger.info(f"Train Monthly Shape: {train_monthly.shape}, Test Monthly Shape: {test_monthly.shape}")

y_train = train_persona[TARGET_COL]
train_ids = train_persona[ID_COL].values
test_ids = test_persona[ID_COL].values

logger.info(f"定着率: {y_train.mean():.4f}")
logger.info(f"Train IDs: {len(train_ids)}, Test IDs: {len(test_ids)}")

[2026-08-09 12:13:13] [INFO] Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


INFO:18_multi_model_expanded_d:Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


[2026-08-09 12:13:13] [INFO] Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


INFO:18_multi_model_expanded_d:Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


[2026-08-09 12:13:13] [INFO] 定着率: 0.5647


INFO:18_multi_model_expanded_d:定着率: 0.5647


[2026-08-09 12:13:13] [INFO] Train IDs: 2761, Test IDs: 2502


INFO:18_multi_model_expanded_d:Train IDs: 2761, Test IDs: 2502


## 1. 基本特徴量関数の定義（split非依存）

In [7]:
def create_monthly_aggregation_features(monthly_df, employee_ids):
    """月次データから集約特徴量を生成（12_〜17_と同一ロジック）"""
    numeric_cols = [
        "残業時間", "有給取得日数", "欠勤日数", "研修時間",
        "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
        "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
        "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
        "顧客満足度評価", "担当プロジェクト数", "月例給与_円"
    ]

    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].copy()
        emp_data = emp_data.sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        for col in numeric_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            valid_values = values[~pd.isna(values)]

            features[f"{col}_mean"] = np.mean(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_std"] = np.std(valid_values) if len(valid_values) > 1 else np.nan
            features[f"{col}_min"] = np.min(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_max"] = np.max(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_median"] = np.median(valid_values) if len(valid_values) > 0 else np.nan
            mean_val = features[f"{col}_mean"]
            std_val = features[f"{col}_std"]
            features[f"{col}_cv"] = std_val / mean_val if (mean_val and mean_val != 0) else np.nan

            early = emp_data[emp_data["経過月数"].between(0, 2)][col]
            mid = emp_data[emp_data["経過月数"].between(3, 11)][col]
            late = emp_data[emp_data["経過月数"].between(12, 23)][col]
            features[f"{col}_early_mean"] = early.mean()
            features[f"{col}_mid_mean"] = mid.mean()
            features[f"{col}_late_mean"] = late.mean()
            features[f"{col}_late_minus_early"] = late.mean() - early.mean()
            features[f"{col}_late_early_ratio"] = (
                late.mean() / early.mean() if early.mean() and early.mean() != 0 else np.nan
            )

            if len(valid_values) >= 2:
                valid_indices = np.where(~pd.isna(values))[0]
                if len(valid_indices) >= 2:
                    slope, _, _, _, _ = stats.linregress(valid_indices, valid_values)
                    features[f"{col}_slope"] = slope
                else:
                    features[f"{col}_slope"] = np.nan
                first_val, last_val = valid_values[0], valid_values[-1]
                features[f"{col}_diff"] = last_val - first_val
                features[f"{col}_ratio"] = last_val / first_val if first_val != 0 else np.nan
            else:
                features[f"{col}_slope"] = features[f"{col}_diff"] = features[f"{col}_ratio"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_monthly_categorical_change_features(monthly_df, employee_ids):
    categorical_cols = ["部署ID", "職種", "役割", "等級", "勤務地", "上司ID"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in categorical_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            changes = sum(1 for i in range(1, len(values)) if pd.notna(values[i]) and pd.notna(values[i-1]) and values[i] != values[i-1])
            features[f"{col}_changes"] = changes
            features[f"{col}_unique_count"] = len(pd.Series(values).dropna().unique())
        if "月末在籍状態" in emp_data.columns:
            status_values = emp_data["月末在籍状態"].values
            features["leave_of_absence_flag"] = int("休職" in status_values)
            features["leave_of_absence_months"] = np.sum(status_values == "休職")
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_missing_value_features(monthly_df, employee_ids):
    missing_target_cols = ["360度評価_親和度", "360度評価_信頼度", "顧客満足度評価", "担当プロジェクト数"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in missing_target_cols:
            if col in emp_data.columns:
                values = emp_data[col].values
                total_months = len(values)
                features[f"{col}_missing_rate"] = pd.isna(values).sum() / total_months if total_months > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_domain_knowledge_features(monthly_df, employee_ids):
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
        eval_mean_list = [emp_data[col].mean() for col in eval_cols if col in emp_data.columns]
        features["engagement_score"] = np.nanmean(eval_mean_list) if len(eval_mean_list) > 0 else np.nan
        if "残業時間" in emp_data.columns:
            features["overtime_stability"] = emp_data["残業時間"].std()
        if "研修時間" in emp_data.columns and "残業時間" in emp_data.columns:
            training_mean = emp_data["研修時間"].mean()
            overtime_mean = emp_data["残業時間"].mean()
            features["training_overtime_ratio"] = training_mean / overtime_mean if overtime_mean > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_advanced_statistical_features(monthly_df, employee_ids):
    """統計的特徴量：歪度、尖度、パーセンタイル"""
    numeric_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in numeric_cols:
            if col in emp_data.columns:
                values = emp_data[col].dropna().values
                if len(values) >= 3:
                    features[f"{col}_skew"] = stats.skew(values)
                    features[f"{col}_kurtosis"] = stats.kurtosis(values)
                    features[f"{col}_q25"] = np.percentile(values, 25)
                    features[f"{col}_q75"] = np.percentile(values, 75)
                    features[f"{col}_iqr"] = features[f"{col}_q75"] - features[f"{col}_q25"]
                else:
                    features[f"{col}_skew"] = features[f"{col}_kurtosis"] = np.nan
                    features[f"{col}_q25"] = features[f"{col}_q75"] = features[f"{col}_iqr"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_cluster_features(monthly_df, employee_ids, n_clusters=5, seed=42):
    """クラスター特徴量：月次データの平均をKMeansクラスタリング"""
    key_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    agg_data = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id]
        row = {"社員ID": employee_id}
        for col in key_cols:
            if col in emp_data.columns:
                row[col] = emp_data[col].mean()
        agg_data.append(row)
    agg_df = pd.DataFrame(agg_data)
    feature_cols = [c for c in key_cols if c in agg_df.columns]
    X = agg_df[feature_cols].fillna(-999)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    kmeans = KMeans(n_clusters=n_clusters, random_state=seed, n_init=10)
    agg_df["cluster"] = kmeans.fit_predict(X_scaled)
    return agg_df[["社員ID", "cluster"]]


def create_eda_driven_features(monthly_df, employee_ids):
    """欠勤日数パターン・360度評価タイミング・月次ボラティリティ・比率特徴量（12_〜17_と同一ロジック）"""
    eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        absence_vals = emp_data["欠勤日数"].values
        nonzero = absence_vals > 0
        features["欠勤発生月数"] = int(nonzero.sum())
        max_run = cur_run = 0
        for v in nonzero:
            cur_run = cur_run + 1 if v else 0
            max_run = max(max_run, cur_run)
        features["欠勤_最長連続月数"] = max_run
        features["欠勤_連続フラグ"] = int(max_run >= 2)

        flagged = emp_data[emp_data["360度評価更新フラグ"] == 1]
        first_month = flagged["経過月数"].min() if len(flagged) > 0 else np.nan
        features["初回評価月"] = first_month
        features["is_早期評価"] = int(first_month <= 4) if pd.notna(first_month) else 0
        features["is_遅延評価"] = int(first_month >= 7) if pd.notna(first_month) else 0
        features["評価遅延度"] = abs(first_month - 5) if pd.notna(first_month) else np.nan

        for col in ["残業時間", "月例給与_円"]:
            vals = emp_data[col].dropna().values
            features[f"{col}_volatility"] = np.mean(np.abs(np.diff(vals))) if len(vals) >= 2 else np.nan

        n_months = len(emp_data)
        features["有給取得率"] = emp_data["有給取得日数"].sum() / n_months if n_months > 0 else np.nan
        features["評価項目間ばらつき"] = emp_data[eval_cols].std(axis=1).mean()

        features_list.append(features)
    return pd.DataFrame(features_list)


def create_manager_team_size_features(monthly_df, employee_ids):
    """初期（経過月数=0）時点で同じ上司IDを持つ社員数（12_〜17_と同一ロジック）"""
    month0 = monthly_df[monthly_df["経過月数"] == 0].copy()
    month0["初期上司_部下数"] = month0.groupby("上司ID")["社員ID"].transform("count")
    out = month0[["社員ID", "初期上司_部下数"]]
    return out[out["社員ID"].isin(employee_ids)].reset_index(drop=True)

print("✅ split非依存の基本特徴量関数定義完了")

✅ split非依存の基本特徴量関数定義完了


In [8]:
logger.info("-" * 60)
logger.info("split非依存の基本特徴量を生成中...")
logger.info("-" * 60)

train_monthly_agg = create_monthly_aggregation_features(train_monthly, train_ids)
test_monthly_agg = create_monthly_aggregation_features(test_monthly, test_ids)

train_cat_change = create_monthly_categorical_change_features(train_monthly, train_ids)
test_cat_change = create_monthly_categorical_change_features(test_monthly, test_ids)

train_missing = create_missing_value_features(train_monthly, train_ids)
test_missing = create_missing_value_features(test_monthly, test_ids)

train_domain = create_domain_knowledge_features(train_monthly, train_ids)
test_domain = create_domain_knowledge_features(test_monthly, test_ids)

train_advanced_stats = create_advanced_statistical_features(train_monthly, train_ids)
test_advanced_stats = create_advanced_statistical_features(test_monthly, test_ids)

train_cluster = create_cluster_features(train_monthly, train_ids, n_clusters=5, seed=SEED)
test_cluster = create_cluster_features(test_monthly, test_ids, n_clusters=5, seed=SEED)

train_eda_feats = create_eda_driven_features(train_monthly, train_ids)
test_eda_feats = create_eda_driven_features(test_monthly, test_ids)

train_mgr = create_manager_team_size_features(train_monthly, train_ids)
test_mgr = create_manager_team_size_features(test_monthly, test_ids)

logger.info("split非依存の基本特徴量生成完了")

[2026-08-09 12:13:13] [INFO] ------------------------------------------------------------


INFO:18_multi_model_expanded_d:------------------------------------------------------------


[2026-08-09 12:13:13] [INFO] split非依存の基本特徴量を生成中...


INFO:18_multi_model_expanded_d:split非依存の基本特徴量を生成中...


[2026-08-09 12:13:13] [INFO] ------------------------------------------------------------


INFO:18_multi_model_expanded_d:------------------------------------------------------------


[2026-08-09 12:20:20] [INFO] split非依存の基本特徴量生成完了


INFO:18_multi_model_expanded_d:split非依存の基本特徴量生成完了


## 2. テキストTF-IDF（A_v1のみ採用）

`17_`のアブレーションで、次元削減バリエーション(A_v2/A_v3)より元の15次元(A_v1)が
2つのsplitで一貫して最良と判明したため、以降はA_v1のみを使う。

In [9]:
def create_tfidf_svd_features(train_persona, test_persona, col, max_features=300, n_components=15, min_df=3, seed=42):
    """文字n-gram TF-IDF + TruncatedSVDでテキスト特徴量を生成（Trainのみでfit）"""
    train_text = train_persona[col].fillna("").astype(str)
    test_text = test_persona[col].fillna("").astype(str)

    vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=max_features, min_df=min_df)
    train_tfidf = vectorizer.fit_transform(train_text)
    test_tfidf = vectorizer.transform(test_text)

    n_comp = min(n_components, train_tfidf.shape[1] - 1)
    svd = TruncatedSVD(n_components=n_comp, random_state=seed, algorithm="arpack")
    train_svd = svd.fit_transform(train_tfidf)
    test_svd = svd.transform(test_tfidf)

    col_names = [f"{col}_tfidf_svd_{i}" for i in range(n_comp)]
    train_out = pd.DataFrame(train_svd, columns=col_names)
    train_out[ID_COL] = train_persona[ID_COL].values
    test_out = pd.DataFrame(test_svd, columns=col_names)
    test_out[ID_COL] = test_persona[ID_COL].values
    return train_out, test_out, svd.explained_variance_ratio_.sum()

TEXT_COLS = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック"]

logger.info("テキストTF-IDF+SVD特徴量(A_v1)を生成中...")
tfidf_train_list, tfidf_test_list = [], []
for col in TEXT_COLS:
    tr, te, explained_var = create_tfidf_svd_features(train_persona, test_persona, col, max_features=300, n_components=15, min_df=3, seed=SEED)
    logger.info(f"{col}: SVD累積寄与率={explained_var:.3f}")
    tfidf_train_list.append(tr)
    tfidf_test_list.append(te)

logger.info("テキストTF-IDF+SVD特徴量生成完了")

[2026-08-09 12:20:20] [INFO] テキストTF-IDF+SVD特徴量(A_v1)を生成中...


INFO:18_multi_model_expanded_d:テキストTF-IDF+SVD特徴量(A_v1)を生成中...


[2026-08-09 12:20:22] [INFO] 入社時メモ: SVD累積寄与率=0.760


INFO:18_multi_model_expanded_d:入社時メモ: SVD累積寄与率=0.760


[2026-08-09 12:20:26] [INFO] 上司からのフィードバック: SVD累積寄与率=0.360


INFO:18_multi_model_expanded_d:上司からのフィードバック: SVD累積寄与率=0.360


[2026-08-09 12:20:28] [INFO] 同僚からのフィードバック: SVD累積寄与率=0.422


INFO:18_multi_model_expanded_d:同僚からのフィードバック: SVD累積寄与率=0.422


[2026-08-09 12:20:28] [INFO] テキストTF-IDF+SVD特徴量生成完了


INFO:18_multi_model_expanded_d:テキストTF-IDF+SVD特徴量生成完了


## 3. 四半期/加速度特徴量（D）: 元の6指標 vs 拡張16指標

`D_original`（17_と同じ6指標）と、月次データの全16指標に拡張した`D_expanded`の両方を用意し、
後段でどちらが良いか2つのsplitで比較する。

In [10]:
def create_quarterly_features(monthly_df, employee_ids, metrics, suffix=""):
    quarters = {"q1": (0, 5), "q2": (6, 11), "q3": (12, 17), "q4": (18, 23)}
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数")
        features = {"社員ID": employee_id}
        for metric in metrics:
            q_means = {}
            for qname, (lo, hi) in quarters.items():
                vals = emp_data[emp_data["経過月数"].between(lo, hi)][metric]
                q_means[qname] = vals.mean()
                features[f"{metric}_{qname}_mean{suffix}"] = q_means[qname]
            first_half_delta = q_means["q2"] - q_means["q1"] if pd.notna(q_means["q1"]) and pd.notna(q_means["q2"]) else np.nan
            second_half_delta = q_means["q4"] - q_means["q3"] if pd.notna(q_means["q3"]) and pd.notna(q_means["q4"]) else np.nan
            features[f"{metric}_acceleration{suffix}"] = (
                second_half_delta - first_half_delta if pd.notna(first_half_delta) and pd.notna(second_half_delta) else np.nan
            )
        features_list.append(features)
    return pd.DataFrame(features_list)

D_ORIGINAL_METRICS = ["残業時間", "欠勤日数", "月例給与_円", "360度評価_親和度", "有給取得日数", "研修時間"]
D_EXPANDED_METRICS = [
    "残業時間", "有給取得日数", "欠勤日数", "研修時間",
    "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
    "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
    "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
    "顧客満足度評価", "担当プロジェクト数", "月例給与_円",
]

logger.info("四半期/加速度特徴量(D_original: 6指標)を生成中...")
train_quarterly_orig = create_quarterly_features(train_monthly, train_ids, D_ORIGINAL_METRICS)
test_quarterly_orig = create_quarterly_features(test_monthly, test_ids, D_ORIGINAL_METRICS)
logger.info(f"D_original: Train {train_quarterly_orig.shape}, Test {test_quarterly_orig.shape}")

logger.info("四半期/加速度特徴量(D_expanded: 16指標)を生成中...")
train_quarterly_exp = create_quarterly_features(train_monthly, train_ids, D_EXPANDED_METRICS, suffix="_exp")
test_quarterly_exp = create_quarterly_features(test_monthly, test_ids, D_EXPANDED_METRICS, suffix="_exp")
logger.info(f"D_expanded: Train {train_quarterly_exp.shape}, Test {test_quarterly_exp.shape}")

[2026-08-09 12:20:28] [INFO] 四半期/加速度特徴量(D_original: 6指標)を生成中...


INFO:18_multi_model_expanded_d:四半期/加速度特徴量(D_original: 6指標)を生成中...


[2026-08-09 12:21:47] [INFO] D_original: Train (2761, 31), Test (2502, 31)


INFO:18_multi_model_expanded_d:D_original: Train (2761, 31), Test (2502, 31)


[2026-08-09 12:21:47] [INFO] 四半期/加速度特徴量(D_expanded: 16指標)を生成中...


INFO:18_multi_model_expanded_d:四半期/加速度特徴量(D_expanded: 16指標)を生成中...


[2026-08-09 12:24:29] [INFO] D_expanded: Train (2761, 81), Test (2502, 81)


INFO:18_multi_model_expanded_d:D_expanded: Train (2761, 81), Test (2502, 81)


## 4. Persona単位の基本特徴量（split非依存）

In [11]:
logger.info("Persona単位の基本特徴量を生成中...")
train_persona["入社日"] = pd.to_datetime(train_persona["入社日"])
test_persona["入社日"] = pd.to_datetime(test_persona["入社日"])

for col in TEXT_COLS:
    train_persona[f"{col}_len"] = train_persona[col].fillna("").astype(str).apply(len)
    test_persona[f"{col}_len"] = test_persona[col].fillna("").astype(str).apply(len)
train_persona["text_total_chars"] = train_persona[TEXT_COLS].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)
test_persona["text_total_chars"] = test_persona[TEXT_COLS].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)

train_persona["入社年"] = train_persona["入社日"].dt.year
train_persona["入社月"] = train_persona["入社日"].dt.month
train_persona["入社四半期"] = train_persona["入社日"].dt.quarter
test_persona["入社年"] = test_persona["入社日"].dt.year
test_persona["入社月"] = test_persona["入社日"].dt.month
test_persona["入社四半期"] = test_persona["入社日"].dt.quarter

train_persona["年齢_x_前職経験"] = train_persona["入社時年齢"] * train_persona["前職経験月数"]
test_persona["年齢_x_前職経験"] = test_persona["入社時年齢"] * test_persona["前職経験月数"]
grade_map = {"G1": 1, "G2": 2, "G3": 3, "G4": 4, "G5": 5}
train_persona["初期等級_num"] = train_persona["初期等級"].map(grade_map)
test_persona["初期等級_num"] = test_persona["初期等級"].map(grade_map)
train_persona["初任給_x_等級"] = train_persona["初任給_円"] * train_persona["初期等級_num"]
test_persona["初任給_x_等級"] = test_persona["初任給_円"] * test_persona["初期等級_num"]

train_persona["is_Q2_新卒"] = ((train_persona["入社四半期"] == 2) & (train_persona["入社区分"] == "新卒")).astype(int)
test_persona["is_Q2_新卒"] = ((test_persona["入社四半期"] == 2) & (test_persona["入社区分"] == "新卒")).astype(int)

logger.info("Persona単位の基本特徴量処理完了")

[2026-08-09 12:24:29] [INFO] Persona単位の基本特徴量を生成中...


INFO:18_multi_model_expanded_d:Persona単位の基本特徴量を生成中...


[2026-08-09 12:24:29] [INFO] Persona単位の基本特徴量処理完了


INFO:18_multi_model_expanded_d:Persona単位の基本特徴量処理完了


## 5. 部署Target Encoding（リーク対策済）と `prepare_split` 関数

`d_variant`パラメータで `None`/`"original"`/`"expanded"` を切り替えられるようにする。

In [12]:
def create_department_target_encoding(train_persona, test_persona, y_train, fit_ids, seed=42, n_splits=5, smoothing=10):
    """初期部署IDのKFold + スムージング付きTarget Encoding（15_〜17_の修正版と同一ロジック）"""
    col = "初期部署ID"
    is_fit = train_persona[ID_COL].isin(fit_ids).values
    dept_all = train_persona[col].values
    y_arr = y_train.values
    global_mean = y_arr[is_fit].mean()

    fit_indices = np.where(is_fit)[0]
    dept_fit = dept_all[fit_indices]
    y_fit = y_arr[fit_indices]

    train_te = np.full(len(train_persona), global_mean)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr_idx, val_idx in kf.split(fit_indices):
        df_tr = pd.DataFrame({col: dept_fit[tr_idx], "y": y_fit[tr_idx]})
        stats_tr = df_tr.groupby(col)["y"].agg(["mean", "count"])
        smoothed = (stats_tr["count"] * stats_tr["mean"] + smoothing * global_mean) / (stats_tr["count"] + smoothing)
        mapping = smoothed.to_dict()
        actual_val_idx = fit_indices[val_idx]
        train_te[actual_val_idx] = pd.Series(dept_fit[val_idx]).map(mapping).fillna(global_mean).values

    df_full = pd.DataFrame({col: dept_fit, "y": y_fit})
    stats_full = df_full.groupby(col)["y"].agg(["mean", "count"])
    smoothed_full = (stats_full["count"] * stats_full["mean"] + smoothing * global_mean) / (stats_full["count"] + smoothing)
    mapping_full = smoothed_full.to_dict()
    dept_size_map = stats_full["count"].to_dict()

    not_fit_indices = np.where(~is_fit)[0]
    train_te[not_fit_indices] = pd.Series(dept_all[not_fit_indices]).map(mapping_full).fillna(global_mean).values

    test_te = test_persona[col].map(mapping_full).fillna(global_mean).values

    train_out = pd.DataFrame({
        ID_COL: train_persona[ID_COL].values,
        "dept_target_enc": train_te,
        "dept_size": pd.Series(dept_all).map(dept_size_map).fillna(0).values,
    })
    test_out = pd.DataFrame({
        ID_COL: test_persona[ID_COL].values,
        "dept_target_enc": test_te,
        "dept_size": test_persona[col].map(dept_size_map).fillna(0).values,
    })
    return train_out, test_out


def prepare_split(split_ratio, d_variant=None):
    """指定した分割比率で特徴量を組み立てる。d_variant: None/"original"/"expanded" """
    sorted_persona = train_persona.sort_values("入社日")
    split_point = int(len(sorted_persona) * split_ratio)
    train_period_ids = set(sorted_persona.iloc[:split_point][ID_COL])

    train_dept_te, test_dept_te = create_department_target_encoding(
        train_persona, test_persona, y_train, fit_ids=train_period_ids, seed=SEED, n_splits=5, smoothing=10
    )

    train_persona_features = train_persona.drop(columns=[TARGET_COL])
    tf = train_persona_features.merge(train_monthly_agg, on=ID_COL, how="left")
    tf = tf.merge(train_cat_change, on=ID_COL, how="left")
    tf = tf.merge(train_missing, on=ID_COL, how="left")
    tf = tf.merge(train_domain, on=ID_COL, how="left")
    tf = tf.merge(train_advanced_stats, on=ID_COL, how="left")
    tf = tf.merge(train_cluster, on=ID_COL, how="left")
    tf = tf.merge(train_dept_te, on=ID_COL, how="left")
    tf = tf.merge(train_eda_feats, on=ID_COL, how="left")
    tf = tf.merge(train_mgr, on=ID_COL, how="left")
    for trdf in tfidf_train_list:
        tf = tf.merge(trdf, on=ID_COL, how="left")

    ttf = test_persona.merge(test_monthly_agg, on=ID_COL, how="left")
    ttf = ttf.merge(test_cat_change, on=ID_COL, how="left")
    ttf = ttf.merge(test_missing, on=ID_COL, how="left")
    ttf = ttf.merge(test_domain, on=ID_COL, how="left")
    ttf = ttf.merge(test_advanced_stats, on=ID_COL, how="left")
    ttf = ttf.merge(test_cluster, on=ID_COL, how="left")
    ttf = ttf.merge(test_dept_te, on=ID_COL, how="left")
    ttf = ttf.merge(test_eda_feats, on=ID_COL, how="left")
    ttf = ttf.merge(test_mgr, on=ID_COL, how="left")
    for tedf in tfidf_test_list:
        ttf = ttf.merge(tedf, on=ID_COL, how="left")

    if d_variant == "original":
        tf = tf.merge(train_quarterly_orig, on=ID_COL, how="left")
        ttf = ttf.merge(test_quarterly_orig, on=ID_COL, how="left")
    elif d_variant == "expanded":
        tf = tf.merge(train_quarterly_exp, on=ID_COL, how="left")
        ttf = ttf.merge(test_quarterly_exp, on=ID_COL, how="left")

    _train_period_features = tf[tf[ID_COL].isin(train_period_ids)]
    job_dev_metrics = ["残業時間_mean", "研修時間_mean", "360度評価_親和度_mean"]
    job_means = {m: _train_period_features.groupby("初期職種")[m].mean().to_dict() for m in job_dev_metrics}
    category_means_train = {m: _train_period_features.groupby("入社区分")[m].mean().to_dict() for m in job_dev_metrics}
    grade_salary_mean = _train_period_features.groupby("初期等級")["初任給_円"].mean().to_dict()
    category_salary_mean = _train_period_features.groupby("入社区分")["初任給_円"].mean().to_dict()
    grade_monthly_salary_mean = _train_period_features.groupby("初期等級")["月例給与_円_mean"].mean().to_dict()

    for df in [tf, ttf]:
        for m in job_dev_metrics:
            df[f"{m}_job_deviation"] = df[m] - df["初期職種"].map(job_means[m])
        df["研修時間_職種比"] = df["研修時間_mean"] / df["初期職種"].map(job_means["研修時間_mean"]).replace(0, np.nan)
        df["研修時間_区分比"] = df["研修時間_mean"] / df["入社区分"].map(category_means_train["研修時間_mean"]).replace(0, np.nan)
        df["初任給_等級内偏差"] = df["初任給_円"] - df["初期等級"].map(grade_salary_mean)
        df["初任給_区分内偏差"] = df["初任給_円"] - df["入社区分"].map(category_salary_mean)
        df["月例給与_等級内偏差"] = df["月例給与_円_mean"] - df["初期等級"].map(grade_monthly_salary_mean)

    drop_cols = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック",
                 "初期部署ID", "初期等級", "最終学歴", "前職職種"]
    tf = tf.drop(columns=[c for c in drop_cols if c in tf.columns]).set_index(ID_COL)
    ttf = ttf.drop(columns=[c for c in drop_cols if c in ttf.columns]).set_index(ID_COL)

    target_series = train_persona.set_index(ID_COL)[TARGET_COL]
    tf_sorted = tf.sort_values("入社日")
    y_sorted = target_series.loc[tf_sorted.index]

    ag_train = tf_sorted.iloc[:split_point].copy()
    ag_tuning = tf_sorted.iloc[split_point:].copy()
    ag_train[TARGET_COL] = y_sorted.iloc[:split_point].values
    ag_tuning[TARGET_COL] = y_sorted.iloc[split_point:].values

    return ag_train, ag_tuning, ttf

print("✅ 部署Target Encoding・prepare_split関数定義完了")

✅ 部署Target Encoding・prepare_split関数定義完了


## 6. チェックポイント機能

計算済みの設定はスキップし、未計算の設定のみ実行する。接続切断からの再開を容易にする。

In [13]:
def load_checkpoint():
    if CHECKPOINT_PATH.exists():
        return pd.read_csv(CHECKPOINT_PATH)
    return pd.DataFrame(columns=["config", "n_features", "val_score", "submission_path"])

def save_checkpoint_row(result):
    df = pd.DataFrame([result])
    write_header = not CHECKPOINT_PATH.exists()
    df.to_csv(CHECKPOINT_PATH, mode="a", header=write_header, index=False)

def run_or_resume(config_label, run_fn):
    checkpoint = load_checkpoint()
    existing = checkpoint[checkpoint["config"] == config_label]
    if len(existing) > 0:
        row = existing.iloc[0].to_dict()
        logger.info(f"[{config_label}] チェックポイントから復元: val_score={row['val_score']:.6f}")
        return row
    result = run_fn()
    save_checkpoint_row(result)
    return result

print("✅ チェックポイント関数定義完了")

✅ チェックポイント関数定義完了


## 7. モデル実行関数（CatBoost / LightGBM / XGBoost）

単一時系列ホールドアウト・Optuna探索という共通の枠組みで、モデル種別ごとにパラメータ空間と
カテゴリ変数の扱いだけを切り替える。

In [14]:
def run_model_config(model_type, ag_train_data, ag_tuning_data, test_features, config_label, n_trials=25):
    feature_cols = [c for c in ag_train_data.columns if c not in ["入社日", TARGET_COL]]
    obj_cols = [c for c in feature_cols if ag_train_data[c].dtype == "object"]

    X_tr_raw = ag_train_data[feature_cols].fillna(-999)
    y_tr = ag_train_data[TARGET_COL]
    X_va_raw = ag_tuning_data[feature_cols].fillna(-999)
    y_va = ag_tuning_data[TARGET_COL]
    X_test_raw = test_features[feature_cols].fillna(-999)

    if model_type == "catboost":
        X_tr, X_va, X_test = X_tr_raw, X_va_raw, X_test_raw

        def objective(trial):
            params = {
                "depth": trial.suggest_int("depth", 3, 10),
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
                "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
                "border_count": trial.suggest_int("border_count", 32, 255),
                "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
                "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
                "iterations": 1000, "random_seed": SEED, "verbose": False,
                "cat_features": obj_cols, "early_stopping_rounds": 50, "task_type": "GPU",
            }
            model = cb.CatBoostClassifier(**params)
            model.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True)
            return log_loss(y_va, model.predict_proba(X_va)[:, 1])

    elif model_type == "lightgbm":
        X_tr, X_va, X_test = X_tr_raw.copy(), X_va_raw.copy(), X_test_raw.copy()
        for df in [X_tr, X_va, X_test]:
            for c in obj_cols:
                df[c] = df[c].astype("category")

        def objective(trial):
            params = {
                "objective": "binary", "metric": "binary_logloss",
                "num_leaves": trial.suggest_int("num_leaves", 15, 127),
                "max_depth": trial.suggest_int("max_depth", 3, 10),
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
                "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
                "subsample": trial.suggest_float("subsample", 0.5, 1.0),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
                "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
                "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
                "verbose": -1, "seed": SEED, "device": "gpu",
            }
            trn = lgb.Dataset(X_tr, label=y_tr)
            val = lgb.Dataset(X_va, label=y_va, reference=trn)
            model = lgb.train(params, trn, num_boost_round=1000, valid_sets=[val],
                               callbacks=[lgb.early_stopping(50, verbose=False)])
            return log_loss(y_va, model.predict(X_va))

    elif model_type == "xgboost":
        X_tr, X_va, X_test = X_tr_raw.copy(), X_va_raw.copy(), X_test_raw.copy()
        for df in [X_tr, X_va, X_test]:
            for c in obj_cols:
                df[c] = df[c].astype("category")

        def objective(trial):
            params = {
                "objective": "binary:logistic", "eval_metric": "logloss",
                "max_depth": trial.suggest_int("max_depth", 3, 10),
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
                "subsample": trial.suggest_float("subsample", 0.5, 1.0),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
                "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
                "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
                "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
                "seed": SEED, "enable_categorical": True, "tree_method": "hist", "device": "cuda",
            }
            dtrain = xgb.DMatrix(X_tr, label=y_tr, enable_categorical=True)
            dval = xgb.DMatrix(X_va, label=y_va, enable_categorical=True)
            model = xgb.train(params, dtrain, num_boost_round=1000, evals=[(dval, "val")],
                               early_stopping_rounds=50, verbose_eval=False)
            return log_loss(y_va, model.predict(dval))
    else:
        raise ValueError(f"unknown model_type: {model_type}")

    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=n_trials)
    best_params = study.best_params

    # ベストパラメータで深い再学習
    if model_type == "catboost":
        final_model = cb.CatBoostClassifier(
            **best_params, iterations=3000, random_seed=SEED, verbose=False,
            cat_features=obj_cols, early_stopping_rounds=100, task_type="GPU",
        )
        final_model.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True)
        val_preds = final_model.predict_proba(X_va)[:, 1]
        test_preds = final_model.predict_proba(X_test)[:, 1]
    elif model_type == "lightgbm":
        params = {**best_params, "objective": "binary", "metric": "binary_logloss", "verbose": -1, "seed": SEED, "device": "gpu"}
        trn = lgb.Dataset(X_tr, label=y_tr)
        val = lgb.Dataset(X_va, label=y_va, reference=trn)
        final_model = lgb.train(params, trn, num_boost_round=3000, valid_sets=[val],
                                 callbacks=[lgb.early_stopping(100, verbose=False)])
        val_preds = final_model.predict(X_va)
        test_preds = final_model.predict(X_test)
    elif model_type == "xgboost":
        params = {**best_params, "objective": "binary:logistic", "eval_metric": "logloss",
                  "seed": SEED, "enable_categorical": True, "tree_method": "hist", "device": "cuda"}
        dtrain = xgb.DMatrix(X_tr, label=y_tr, enable_categorical=True)
        dval = xgb.DMatrix(X_va, label=y_va, enable_categorical=True)
        dtest = xgb.DMatrix(X_test, enable_categorical=True)
        final_model = xgb.train(params, dtrain, num_boost_round=3000, evals=[(dval, "val")],
                                 early_stopping_rounds=100, verbose_eval=False)
        val_preds = final_model.predict(dval)
        test_preds = final_model.predict(dtest)

    val_score = log_loss(y_va, val_preds)
    sub_path = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_{config_label}.csv"
    sub = pd.DataFrame({ID_COL: test_features.index, TARGET_COL: test_preds})
    sub.to_csv(sub_path, index=False, header=False)

    # 検証予測も保存（アンサンブル計算用）
    val_pred_path = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_{config_label}_valpreds.npy"
    np.save(val_pred_path, val_preds)

    logger.info(f"[{config_label}] model={model_type}, n_features={len(feature_cols)}, val_score={val_score:.6f}")
    return {
        "config": config_label, "n_features": len(feature_cols), "val_score": val_score,
        "submission_path": str(sub_path), "val_pred_path": str(val_pred_path),
    }

print("✅ run_model_config関数定義完了")

✅ run_model_config関数定義完了


## 8. ステップA: Dブロック比較（Dなし / D_original / D_expanded）× 2 split

CatBoostで3つの特徴量構成を80/20・75/25の両方で評価し、Dブロックをどちらにするか決める。

In [15]:
SPLIT_RATIOS = {"split_80_20": 0.8, "split_75_25": 0.75}
D_VARIANTS = {"no_d": None, "d_original": "original", "d_expanded": "expanded"}

stepA_results = []
for split_name, ratio in SPLIT_RATIOS.items():
    for d_name, d_variant in D_VARIANTS.items():
        config_label = f"stepA_{split_name}_catboost_{d_name}"
        def _run(ratio=ratio, d_variant=d_variant, config_label=config_label):
            logger.info(f"=== {config_label} ===")
            ag_train_data, ag_tuning_data, test_features_full = prepare_split(ratio, d_variant=d_variant)
            return run_model_config("catboost", ag_train_data, ag_tuning_data, test_features_full, config_label, n_trials=25)
        result = run_or_resume(config_label, _run)
        result["split"] = split_name
        result["d_variant"] = d_name
        stepA_results.append(result)

stepA_df = pd.DataFrame(stepA_results)
stepA_pivot = stepA_df.pivot(index="d_variant", columns="split", values="val_score")
stepA_pivot["mean"] = stepA_pivot.mean(axis=1)
stepA_pivot["std"] = stepA_pivot[["split_80_20", "split_75_25"]].std(axis=1)
stepA_pivot = stepA_pivot.sort_values("mean")

logger.info("=" * 60)
logger.info("ステップA結果（Dブロック比較）")
logger.info("=" * 60)
logger.info("\n" + stepA_pivot.to_string())
print("\n■ ステップA結果:")
print(stepA_pivot.to_string())

best_d_variant_name = stepA_pivot["mean"].idxmin()
best_d_variant = D_VARIANTS[best_d_variant_name]
logger.info(f"選ばれたDブロック構成: {best_d_variant_name}")
print(f"\n■ 選ばれたDブロック構成: {best_d_variant_name}")

[2026-08-09 12:24:30] [INFO] === stepA_split_80_20_catboost_no_d ===


INFO:18_multi_model_expanded_d:=== stepA_split_80_20_catboost_no_d ===


[2026-08-09 12:54:13] [INFO] [stepA_split_80_20_catboost_no_d] model=catboost, n_features=359, val_score=0.533476


INFO:18_multi_model_expanded_d:[stepA_split_80_20_catboost_no_d] model=catboost, n_features=359, val_score=0.533476


[2026-08-09 12:54:13] [INFO] === stepA_split_80_20_catboost_d_original ===


...

[2026-08-09 14:32:28] [INFO] 
split       split_75_25  split_80_20      mean       std
d_variant                                               
d_expanded     0.544731     0.537033  0.540882  0.005443
no_d           0.554675     0.533476  0.544075  0.014990
d_original     0.552516     0.539620  0.546068  0.009118


INFO:18_multi_model_expanded_d:
split       split_75_25  split_80_20      mean       std
d_variant                                               
d_expanded     0.544731     0.537033  0.540882  0.005443
no_d           0.554675     0.533476  0.544075  0.014990
d_original     0.552516     0.539620  0.546068  0.009118



■ ステップA結果:
split       split_75_25  split_80_20      mean       std
d_variant                                               
d_expanded     0.544731     0.537033  0.540882  0.005443
no_d           0.554675     0.533476  0.544075  0.014990
d_original     0.552516     0.539620  0.546068  0.009118
[2026-08-09 14:32:28] [INFO] 選ばれたDブロック構成: d_expanded


INFO:18_multi_model_expanded_d:選ばれたDブロック構成: d_expanded



■ 選ばれたDブロック構成: d_expanded


## 9. ステップB: モデル比較（CatBoost/LightGBM/XGBoost）× 2 split

ステップAで選ばれたDブロック構成を使い、3モデルをそれぞれ80/20・75/25で評価する。
CatBoostの結果はステップAで既に計算済みならチェックポイントから再利用される。

In [16]:
MODEL_TYPES = ["catboost", "lightgbm", "xgboost"]

stepB_results = []
for split_name, ratio in SPLIT_RATIOS.items():
    for model_type in MODEL_TYPES:
        # CatBoost かつ ステップAの最良Dと同じ構成なら、ステップAのラベルと一致させて再利用する
        if model_type == "catboost":
            config_label = f"stepA_{split_name}_catboost_{best_d_variant_name}"
        else:
            config_label = f"stepB_{split_name}_{model_type}_{best_d_variant_name}"

        def _run(ratio=ratio, model_type=model_type, config_label=config_label):
            logger.info(f"=== {config_label} ===")
            ag_train_data, ag_tuning_data, test_features_full = prepare_split(ratio, d_variant=best_d_variant)
            return run_model_config(model_type, ag_train_data, ag_tuning_data, test_features_full, config_label, n_trials=25)

        result = run_or_resume(config_label, _run)
        result["split"] = split_name
        result["model"] = model_type
        stepB_results.append(result)

stepB_df = pd.DataFrame(stepB_results)
stepB_pivot = stepB_df.pivot(index="model", columns="split", values="val_score")
stepB_pivot["mean"] = stepB_pivot.mean(axis=1)
stepB_pivot["std"] = stepB_pivot[["split_80_20", "split_75_25"]].std(axis=1)
stepB_pivot = stepB_pivot.sort_values("mean")

logger.info("=" * 60)
logger.info("ステップB結果（モデル比較）")
logger.info("=" * 60)
logger.info("\n" + stepB_pivot.to_string())
print("\n■ ステップB結果:")
print(stepB_pivot.to_string())

[2026-08-09 14:32:29] [INFO] [stepA_split_80_20_catboost_d_expanded] チェックポイントから復元: val_score=0.537033


INFO:18_multi_model_expanded_d:[stepA_split_80_20_catboost_d_expanded] チェックポイントから復元: val_score=0.537033


[2026-08-09 14:32:29] [INFO] === stepB_split_80_20_lightgbm_d_expanded ===


INFO:18_multi_model_expanded_d:=== stepB_split_80_20_lightgbm_d_expanded ===


[2026-08-09 14:33:12] [INFO] [stepB_split_80_20_lightgbm_d_expanded] model=lightgbm, n_features=439, val_score=0.566355


...

[2026-08-09 14:37:47] [INFO] ============================================================


INFO:18_multi_model_expanded_d:============================================================


[2026-08-09 14:37:47] [INFO] 
split     split_75_25  split_80_20      mean       std
model                                                 
catboost     0.544731     0.537033  0.540882  0.005443
lightgbm     0.588556     0.566355  0.577455  0.015699
xgboost      0.590452     0.569382  0.579917  0.014899


INFO:18_multi_model_expanded_d:
split     split_75_25  split_80_20      mean       std
model                                                 
catboost     0.544731     0.537033  0.540882  0.005443
lightgbm     0.588556     0.566355  0.577455  0.015699
xgboost      0.590452     0.569382  0.579917  0.014899



■ ステップB結果:
split     split_75_25  split_80_20      mean       std
model                                                 
catboost     0.544731     0.537033  0.540882  0.005443
lightgbm     0.588556     0.566355  0.577455  0.015699
xgboost      0.590452     0.569382  0.579917  0.014899


## 10. ステップC: 単純平均アンサンブル（split_80_20）

過去の教訓（Stacking/Optimized Weightedのような学習型アンサンブルは過学習しやすい、12_/13_/14_で確認済み）を踏まえ、
**固定の単純平均のみ**を試す。3モデルの検証予測（`val_pred_path`に保存済み）を使って
アンサンブルの検証Log Lossを計算し、Test予測も単純平均して提出ファイルを作る。

In [17]:
split_80_20_rows = stepB_df[stepB_df["split"] == "split_80_20"].set_index("model")

_, ag_tuning_ref, test_features_ref = prepare_split(0.8, d_variant=best_d_variant)
y_va_ref = ag_tuning_ref[TARGET_COL]

val_preds_by_model = {}
test_preds_by_model = {}
for model_type in MODEL_TYPES:
    row = split_80_20_rows.loc[model_type]
    val_preds_by_model[model_type] = np.load(row["val_pred_path"])
    test_preds_by_model[model_type] = pd.read_csv(row["submission_path"], header=None, names=[ID_COL, TARGET_COL]).set_index(ID_COL)[TARGET_COL]

ensemble_val_preds = np.mean([val_preds_by_model[m] for m in MODEL_TYPES], axis=0)
ensemble_val_score = log_loss(y_va_ref, ensemble_val_preds)

ensemble_test_preds = pd.concat([test_preds_by_model[m] for m in MODEL_TYPES], axis=1).mean(axis=1)
ensemble_sub_path = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_simple_ensemble.csv"
ensemble_sub = pd.DataFrame({ID_COL: ensemble_test_preds.index, TARGET_COL: ensemble_test_preds.values})
ensemble_sub.to_csv(ensemble_sub_path, index=False, header=False)

logger.info(f"[simple_ensemble] val_score={ensemble_val_score:.6f}")
logger.info(f"提出ファイル保存: {ensemble_sub_path}")

print(f"\n■ 単純平均アンサンブル 検証Log Loss: {ensemble_val_score:.6f}")
print(f"■ 個別モデル（参考、split_80_20）:")
print(split_80_20_rows[["val_score"]].to_string())
print(f"■ 提出ファイル: {ensemble_sub_path}")

[2026-08-09 14:37:47] [INFO] [simple_ensemble] val_score=0.549182


INFO:18_multi_model_expanded_d:[simple_ensemble] val_score=0.549182


[2026-08-09 14:37:47] [INFO] 提出ファイル保存: /content/drive/MyDrive/jaggle_2026/data/output/20260809/20260809_18_multi_model_expanded_d_simple_ensemble.csv


INFO:18_multi_model_expanded_d:提出ファイル保存: /content/drive/MyDrive/jaggle_2026/data/output/20260809/20260809_18_multi_model_expanded_d_simple_ensemble.csv



■ 単純平均アンサンブル 検証Log Loss: 0.549182
■ 個別モデル（参考、split_80_20）:
          val_score
model              
catboost   0.537033
lightgbm   0.566355
xgboost    0.569382
■ 提出ファイル: /content/drive/MyDrive/jaggle_2026/data/output/20260809/20260809_18_multi_model_expanded_d_simple_ensemble.csv


## 11. 総合結果

split_80_20における「単体最良モデル」と「単純平均アンサンブル」を比較する。
（学習型アンサンブルが過学習しやすいという教訓が今回も成り立つか、単純平均は単体最良に対して優位かを確認する。）

In [18]:
summary_rows = split_80_20_rows[["val_score", "submission_path"]].reset_index().rename(columns={"model": "config"})
summary_rows = pd.concat([
    summary_rows,
    pd.DataFrame([{"config": "simple_ensemble", "val_score": ensemble_val_score, "submission_path": str(ensemble_sub_path)}]),
]).sort_values("val_score").reset_index(drop=True)

logger.info("=" * 60)
logger.info("総合結果（split_80_20）")
logger.info("=" * 60)
logger.info("\n" + summary_rows.to_string())
print("\n■ 総合結果（split_80_20）:")
print(summary_rows.to_string(index=False))
print(f"\n(参考) 17_ A_v1_plus_quarterly: val 0.533663, Public 0.552540（現時点の最良）")

summary_rows

[2026-08-09 14:37:47] [INFO] ============================================================


INFO:18_multi_model_expanded_d:============================================================


[2026-08-09 14:37:47] [INFO] 総合結果（split_80_20）


INFO:18_multi_model_expanded_d:総合結果（split_80_20）


[2026-08-09 14:37:47] [INFO] ============================================================


INFO:18_multi_model_expanded_d:============================================================


[2026-08-09 14:37:47] [INFO] 
            config  val_score                                                                                                                       submission_path
0         catboost   0.537033  /content/drive/MyDrive/jaggle_2026/data/output/20260809/20260809_18_multi_model_expanded_d_stepA_split_80_20_catboost_d_expanded.csv
1  simple_ensemble   0.549182                        /content/drive/MyDrive/jaggle_2026/data/output/20260809/20260809_18_multi_model_expanded_d_simple_ensemble.csv
2         lightgbm   0.566355  /content/drive/MyDrive/jaggle_2026/data/output/20260809/20260809_18_multi_model_expanded_d_stepB_split_80_20_lightgbm_d_expanded.csv
3          xgboost   0.569382   /content/drive/MyDrive/jaggle_2026/data/output/20260809/20260809_18_multi_model_expanded_d_stepB_split_80_20_xgboost_d_expanded.csv


INFO:18_multi_model_expanded_d:
            config  val_score                                                                                                                       submission_path
0         catboost   0.537033  /content/drive/MyDrive/jaggle_2026/data/output/20260809/20260809_18_multi_model_expanded_d_stepA_split_80_20_catboost_d_expanded.csv
1  simple_ensemble   0.549182                        /content/drive/MyDrive/jaggle_2026/data/output/20260809/20260809_18_multi_model_expanded_d_simple_ensemble.csv
2         lightgbm   0.566355  /content/drive/MyDrive/jaggle_2026/data/output/20260809/20260809_18_multi_model_expanded_d_stepB_split_80_20_lightgbm_d_expanded.csv
3          xgboost   0.569382   /content/drive/MyDrive/jaggle_2026/data/output/20260809/20260809_18_multi_model_expanded_d_stepB_split_80_20_xgboost_d_expanded.csv



■ 総合結果（split_80_20）:
         config  val_score                                                                                                                      submission_path
       catboost   0.537033 /content/drive/MyDrive/jaggle_2026/data/output/20260809/20260809_18_multi_model_expanded_d_stepA_split_80_20_catboost_d_expanded.csv
simple_ensemble   0.549182                       /content/drive/MyDrive/jaggle_2026/data/output/20260809/20260809_18_multi_model_expanded_d_simple_ensemble.csv
       lightgbm   0.566355 /content/drive/MyDrive/jaggle_2026/data/output/20260809/20260809_18_multi_model_expanded_d_stepB_split_80_20_lightgbm_d_expanded.csv
        xgboost   0.569382  /content/drive/MyDrive/jaggle_2026/data/output/20260809/20260809_18_multi_model_expanded_d_stepB_split_80_20_xgboost_d_expanded.csv

(参考) 17_ A_v1_plus_quarterly: val 0.533663, Public 0.552540（現時点の最良）


,config,val_score,submission_path
0,catboost,0.537033,/content/drive/MyDrive/jaggle_2026/data/output...
1,simple_ensemble,0.549182,/content/drive/MyDrive/jaggle_2026/data/output...
2,lightgbm,0.566355,/content/drive/MyDrive/jaggle_2026/data/output...
3,xgboost,0.569382,/content/drive/MyDrive/jaggle_2026/data/output...


## 12. まとめ・次のアクション

1. Dブロックは`no_d`/`d_original`/`d_expanded`のどれが選ばれたか、ステップAの表で確認する
2. CatBoost/LightGBM/XGBoostのどれが最良か、ステップBの表（mean/std）で確認する
3. 単純平均アンサンブルが単体最良モデルを上回るか確認する（上回らなければ、単体モデルの提出を優先する）
4. 最良の候補（単体最良 or アンサンブル）をKaggleに提出し、Publicスコアを確認する
5. 結果が出たら`submit_result_report.md`に追記する

### バックログ（今回は着手しない）
- テキスト特徴量を日本語の事前学習済み文埋め込みモデルに置き換える案